# HAR Stage 9 — Recover GSE270828 cell metadata without Google Drive

**Verified Stage 8 finding:** the original GEO 2.31-GB file is a two-layer GZIP containing an RDS **Seurat** object, which successfully deserialized in R. The preceding extraction failed only when an S4 class inspection attempted to load a missing `SeuratObject` package. This notebook accesses the `meta.data` S4 attribute using base R, so it does **not** install the full Seurat/SeuratObject dependency tree.

**What it does:** reuse the original file if it remains in this Colab runtime, optionally redownload only if missing, perform the proven double-GZIP streaming read, export cell metadata and guide/replicate QC, and produce a small results ZIP. **No Google Drive access. No differential expression or causal inference.**

Note: If Colab restarted, `/content` will have been reset and the 2.31-GB source must be downloaded again. Keep enough RAM (approximately 11 GB free worked for the earlier deserialization).


In [ ]:
import os, sys, gzip, json, shutil, zipfile, subprocess, re, time, contextlib, hashlib
from pathlib import Path
from collections import Counter
import pandas as pd
import requests

ROOT=Path('/content') if Path('/content').exists() else Path.cwd()
DATA=ROOT/'GSE270828_public_source'
OUT=ROOT/'HAR_Stage9_Output'
DATA.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
ORIGINAL=DATA/'GSE270828_Merged_CountMatrix.rds.gz'
GUIDE=DATA/'GSE270828_feature_README.csv'
META=OUT/'GSE270828_cell_metadata.tsv'
BASE_URL='https://ftp.ncbi.nlm.nih.gov/geo/series/GSE270nnn/GSE270828/suppl/'
ORIGINAL_URL=BASE_URL+'GSE270828_Merged_CountMatrix.rds.gz'
GUIDE_URL=BASE_URL+'GSE270828_feature_README.csv'
EXPECTED_SIZE=2312020968  # As reported by the user's successful original download
STATUS={'project':'HAR perturbation benchmarking','stage':'9','google_drive_used':False,'differential_expression_performed':False,'status':'started'}

def save_status():
    (OUT/'STATUS.json').write_text(json.dumps(STATUS,indent=2,default=str))

def download_resumable(url,path,expected_size=None,retries=4):
    path=Path(path)
    if path.is_file() and (expected_size is None or path.stat().st_size==expected_size):
        return {'status':'reused','bytes':path.stat().st_size}
    partial=path.with_name(path.name+'.part')
    if path.exists() and path.stat().st_size!=expected_size:
        path.rename(partial)
    for attempt in range(1,retries+1):
        offset=partial.stat().st_size if partial.exists() else 0
        h={'Range':f'bytes={offset}-'} if offset else {}
        try:
            with requests.get(url,headers=h,stream=True,timeout=(25,90)) as r:
                r.raise_for_status()
                if offset and r.status_code != 206:
                    # Server ignored Range; rewrite rather than appending a duplicate file.
                    offset=0
                with open(partial,'ab' if offset else 'wb') as f:
                    for piece in r.iter_content(chunk_size=4*1024*1024):
                        if piece:f.write(piece)
            size=partial.stat().st_size
            if expected_size and size!=expected_size:
                raise RuntimeError(f'Incomplete download: {size} bytes; expected {expected_size}')
            partial.rename(path)
            return {'status':'downloaded','bytes':size,'attempts':attempt}
        except Exception as e:
            print('Download attempt',attempt,'failed:',str(e)[:240])
            if attempt==retries:raise
            time.sleep(2*attempt)
    raise RuntimeError('Download failed')

def sniff_layers(path,max_gzip_layers=5):
    # Inspect gzip nesting by reading only tiny prefixes, without full decompression.
    result={'source_exists':Path(path).exists()}
    if not result['source_exists']:return result
    result['compressed_size_bytes']=Path(path).stat().st_size
    with contextlib.ExitStack() as stack:
        stream=stack.enter_context(open(path,'rb'))
        header=stream.read(64)
        stream.seek(0)
        layers=0
        while header.startswith(b'\x1f\x8b') and layers<max_gzip_layers:
            stream=stack.enter_context(gzip.GzipFile(fileobj=stream,mode='rb'))
            header=stream.read(64)
            stream.seek(0)
            layers+=1
        if header.startswith((b'X\n',b'A\n',b'B\n')):
            fmt='RDS'
        elif header.startswith((b'RDX2\n',b'RDA2\n',b'RDB2\n',b'RDX3\n',b'RDA3\n',b'RDB3\n')):
            fmt='R_workspace'
        elif header.startswith(b'\x1f\x8b'):
            fmt='unresolved_additional_gzip'
        else:
            fmt='unrecognized'
        result.update({'gzip_layers':layers,'inner_format':fmt,'inner_header_hex':header[:32].hex(),'inner_header_repr':repr(header[:32])})
        return result

print('Google Drive:', 'NOT USED')
print('Original data in current runtime:', ORIGINAL.exists())
print('Free disk:',round(shutil.disk_usage(ROOT).free/1024**3,2),'GB')
print('Python:',sys.version.split()[0])

In [ ]:
# Configuration: all files remain on /content; no drive mount.
DOWNLOAD_ORIGINAL_IF_MISSING=True    # Change to False to run diagnostics without 2.31 GB download
ATTEMPT_R_METADATA_EXTRACTION=True   # Set False for source-format-only check
MIN_AVAILABLE_MEMORY_GB=6.0         # If less, notebook safely skips R extraction

try:
    if ORIGINAL.exists() and ORIGINAL.stat().st_size==EXPECTED_SIZE:
        STATUS['download']={'status':'reused_existing','bytes':ORIGINAL.stat().st_size}
    elif DOWNLOAD_ORIGINAL_IF_MISSING:
        if shutil.disk_usage(ROOT).free < EXPECTED_SIZE + 3*1024**3:
            raise RuntimeError('Insufficient disk space to download source plus safety margin')
        STATUS['download']=download_resumable(ORIGINAL_URL,ORIGINAL,EXPECTED_SIZE)
    else:
        STATUS['download']={'status':'not_attempted','reason':'DOWNLOAD_ORIGINAL_IF_MISSING=False'}
except Exception as e:
    STATUS['download']={'status':'error','message':str(e)}
print('Source status:',STATUS['download'])
save_status()

In [ ]:
# Verify the INNER serialization header: RDS vs R workspace vs unknown.
try:
    signature=sniff_layers(ORIGINAL)
    STATUS['nested_compression']=signature
except Exception as e:
    STATUS['nested_compression']={'status':'error','error':repr(e)}
(OUT/'nested_gzip_signature.json').write_text(json.dumps(STATUS['nested_compression'],indent=2))
print(json.dumps(STATUS['nested_compression'],indent=2))
save_status()
if STATUS['nested_compression'].get('inner_format') not in ['RDS','R_workspace']:
    print('NOT a verified RDS/workspace yet: R extraction will be blocked.')

## Metadata recovery using base R
Direct `attr(obj, "meta.data", exact=TRUE)` access avoids the missing `SeuratObject` package. R deserialization still requires sufficient memory.


In [ ]:
# Automatically ensure R is available (Google Drive is NOT needed).
# Fresh Python Colab runtimes may not have Rscript installed by default.
import shutil, subprocess
info=STATUS.get('nested_compression',{})
if ATTEMPT_R_METADATA_EXTRACTION and info.get('inner_format') in ('RDS','R_workspace') and not shutil.which('Rscript'):
    print('Rscript not present. Installing minimal R runtime...')
    try:
        subprocess.run(['apt-get','update','-qq'],check=True,timeout=900,stdout=subprocess.DEVNULL)
        subprocess.run(['apt-get','install','-y','-qq','r-base-core'],check=True,timeout=1200,stdout=subprocess.DEVNULL)
        STATUS['r_install']={'status':'installed','rscript_path':shutil.which('Rscript')}
    except Exception as e:
        STATUS['r_install']={'status':'failed','message':str(e)}
else:
    STATUS['r_install']={'status':'already_available_or_not_needed','rscript_path':shutil.which('Rscript')}
print('R installation:',STATUS['r_install'])
save_status()

In [ ]:
# Stage 9: base-R Seurat metadata extraction with no SeuratObject install.
# R source is written as plain text and saved in the output package.
R_SCRIPT=OUT/'stream_extract_metadata_baseR.R'
R_SCRIPT.write_text("args <- commandArgs(trailingOnly=TRUE)\nif (length(args) < 5L) stop('Expected arguments: path, output_tsv, progress_log, gzip_layers, serialization_format')\nsource_path <- args[[1]]\nout_path <- args[[2]]\nlog_path <- args[[3]]\nlayers <- as.integer(args[[4]])\nfmt <- args[[5]]\n\nlogline <- function(...) cat(format(Sys.time(), '%Y-%m-%d %H:%M:%S'), ..., '\\n', file=log_path, append=TRUE)\nlogline('START', 'gzip_layers', layers, 'format', fmt)\nif (is.na(layers) || layers < 1L || layers > 5L) stop('Invalid GZIP layer count')\nif (!file.exists(source_path)) stop('Original source file is missing')\n\n# The source is known to be 2-layer GZIP containing an RDS. Streaming avoids\n# writing another multi-GB uncompressed copy, but the R object must fit in RAM.\nquoted_path <- shQuote(normalizePath(source_path))\ncommands <- c(paste('gzip -dc', quoted_path), rep('gzip -dc', layers - 1L))\ncommand <- paste(commands, collapse=' | ')\ncon <- pipe(command, open='rb')\nlogline('DESERIALIZE_BEGIN')\nif (fmt == 'RDS') {\n  obj <- readRDS(con)\n} else if (fmt == 'R_workspace') {\n  env <- new.env(parent=globalenv())\n  loaded <- load(con, envir=env)\n  if (!length(loaded)) stop('R workspace is empty')\n  scores <- vapply(loaded, function(nm) {\n    candidate <- get(nm, envir=env)\n    !is.null(attr(candidate, 'meta.data', exact=TRUE)) ||\n      (is.list(candidate) && is.data.frame(candidate$meta.data)) ||\n      is.data.frame(candidate)\n  }, logical(1))\n  obj <- get(loaded[[if (any(scores)) which(scores)[[1]] else 1L]], envir=env)\n} else stop('Unexpected serialization format')\ntry(close(con), silent=TRUE)\nlogline('DESERIALIZED', 'class', paste(class(obj), collapse=';'))\n\n# IMPORTANT: do NOT use methods::slotNames(), methods::is(), or Seurat's API\n# here. They require the SeuratObject R package, which is not always installed\n# in a fresh Colab runtime. S4 slots are stored as object attributes; reading\n# the meta.data slot with base::attr() avoids loading SeuratObject.\nmd <- tryCatch(attr(obj, 'meta.data', exact=TRUE), error=function(e) NULL)\nif (is.null(md)) md <- tryCatch(obj@meta.data, error=function(e) NULL)\nif (is.null(md) && is.list(obj)) md <- obj$meta.data\nif (is.null(md) && is.data.frame(obj)) md <- obj\nif (is.null(md) && is.list(obj)) md <- obj$obs\nif (!is.data.frame(md)) stop('Object deserialized but cell metadata data.frame could not be accessed')\nlogline('METADATA_FOUND', 'rows', nrow(md), 'columns', ncol(md))\nif (nrow(md) == 0L) stop('Metadata has zero cell rows')\n\nwanted <- c('orig.ident','lane','rep','replicate','batch','guide','har',\n           'perturbation_name','num_guide','nCount_RNA','nFeature_RNA',\n           'percent.mt','scDblFinder.class','scDblFinder.score')\nadditional <- grep('rep|batch|lane|guide|crispr|mt|sample|donor|har|perturb',\n                   names(md), ignore.case=TRUE, value=TRUE)\nkeep <- unique(c(intersect(wanted, names(md)), additional))\nif (!length(keep)) stop('Metadata exists, but expected experimental-design columns are missing')\nmd <- md[, keep, drop=FALSE]\nmd$cell_id <- rownames(md)\nmd <- md[, c('cell_id', setdiff(names(md), 'cell_id')), drop=FALSE]\nfor (n in names(md)) {\n  if (is.list(md[[n]])) md[[n]] <- vapply(md[[n]], function(x) paste(x,collapse=';'), character(1))\n  if (is.factor(md[[n]])) md[[n]] <- as.character(md[[n]])\n}\n# Ensure this metadata-only step cannot be mistaken for DE gene expression.\nwrite.table(md, file=out_path, sep='\\t', quote=TRUE, row.names=FALSE, na='')\nlogline('SUCCESS', 'metadata_tsv', out_path, 'bytes', file.info(out_path)$size)\n",encoding='utf-8')
# Avoid a stale TSV being treated as a successful fresh extraction.
if META.exists(): META.unlink()
info=STATUS.get('nested_compression',{})
if not ATTEMPT_R_METADATA_EXTRACTION:
    STATUS['r_metadata']={'status':'skipped_by_user'}
elif info.get('inner_format') not in ('RDS','R_workspace'):
    STATUS['r_metadata']={'status':'blocked','reason':'The inner R serialization header is not verified'}
elif not shutil.which('Rscript'):
    STATUS['r_metadata']={'status':'blocked','reason':'Rscript unavailable (requires R runtime)'}
else:
    try:
        mem_gb=None
        try:
            import psutil
            mem_gb=psutil.virtual_memory().available/1024**3
        except ImportError:
            pass
        STATUS['memory_available_gb_before_R']=round(mem_gb,2) if mem_gb else None
        if mem_gb is not None and mem_gb < MIN_AVAILABLE_MEMORY_GB:
            raise MemoryError(f'Only {mem_gb:.1f} GB RAM free; skip heavy R deserialization')
        cmd=['Rscript','--vanilla',str(R_SCRIPT),str(ORIGINAL),str(META),str(OUT/'R_extraction_progress.txt'),
             str(info['gzip_layers']),info['inner_format']]
        print('Reading nested gzip stream with R...')
        with open(OUT/'R_stdout_stderr.txt','w') as log:
            p=subprocess.run(cmd,stdout=log,stderr=subprocess.STDOUT,timeout=7200)
        if p.returncode !=0 or not META.is_file():
            raise RuntimeError(f'R extraction failed with exit code {p.returncode}; see R_stdout_stderr.txt and R_extraction_progress.txt')
        STATUS['r_metadata']={'status':'complete','tsv_bytes':META.stat().st_size}
    except Exception as e:
        STATUS['r_metadata']={'status':'error','message':str(e)}
print('R extraction:',STATUS['r_metadata'])
save_status()

In [ ]:
# Metadata quality-control with no invalid claims of biological independence.
try:
    if not META.exists():
        STATUS['cell_qc']={'status':'not_run','reason':'Cell metadata extraction is unavailable'}
    else:
        t=pd.read_csv(META,sep='\t',low_memory=False)
        required=['cell_id']
        if not all(x in t.columns for x in required):
            raise RuntimeError('Metadata missing cell_id')
        if t['cell_id'].duplicated().any():
            raise RuntimeError('Duplicate cell identifiers in extracted metadata')
        qc={'status':'complete','n_cell_rows':len(t),'available_columns':list(t.columns),
            'unique_cell_barcodes':int(t['cell_id'].nunique()),
            'warning':'Lanes, cells, and sequencing libraries are not independent biological donors.'}
        for col in ['rep','replicate','lane','batch','guide','har','perturbation_name','num_guide','scDblFinder.class']:
            if col in t:
                ct=t[col].fillna('<missing>').astype(str).value_counts().rename_axis(col).reset_index(name='n_cells')
                ct.to_csv(OUT/f'QC_{col}_counts.csv',index=False)
                qc[f'unique_{col}']=int(t[col].nunique(dropna=True))
        if 'har' in t and any(x in t for x in ['rep','replicate']):
            rep='replicate' if 'replicate' in t else 'rep'
            pd.crosstab(t['har'].fillna('<missing>'),t[rep].fillna('<missing>')).to_csv(OUT/'QC_HAR_by_rep.csv')
        qc['guide_level_evidence'] = 'cell assignment only; no transcriptomic differential expression in Stage 9'
        STATUS['cell_qc']=qc
        (OUT/'QC_cell_summary.json').write_text(json.dumps(qc,indent=2))
        print('Cell-level metadata QC complete:',len(t),'rows')
except Exception as e:
    STATUS['cell_qc']={'status':'error','message':str(e)}
print(json.dumps(STATUS['cell_qc'],indent=2)[:4000])
save_status()

In [ ]:
# Small export: source large file is NEVER included.
STATUS['status']='finished_with_metadata' if STATUS.get('cell_qc',{}).get('status')=='complete' else 'finished_diagnostics_only'
save_status()
(OUT/'README.txt').write_text('GSE270828 Stage 9: nested gzip diagnosis and cell-level metadata QC only. No differential expression performed. No Google Drive used. Original multi-GB file is not included.\n')
ARCHIVE=ROOT/'HAR_Stage9_Cell_Metadata_QC_Results.zip'
with zipfile.ZipFile(ARCHIVE,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for f in OUT.rglob('*'):
        if f.is_file() and f!=ORIGINAL:z.write(f,arcname=f.relative_to(OUT).as_posix())
print('Output ZIP:',ARCHIVE,ARCHIVE.stat().st_size,'bytes')
try:
    from google.colab import files
    files.download(str(ARCHIVE))
except ImportError:
    print('Download ZIP from local notebook folder:',ARCHIVE)
except Exception as e:
    print('Auto download did not start:',e,'; use Colab Files panel')
print('Stage 9 status:',STATUS['status'])
